# 手順3e：キャビティ厚み H を入力にすることの検討

ユーザーの問い（2026-10-14）：**H は、目的変数（見かけの粘度）を算出するときの流量の計算に使われる値なので、入力に使うとリークではないか。Cross-WLF モデルの入力（せん断速度・温度・圧力）には厚みがないので、厚みを入力にする学習方法には疑問がある。**

この手順では、次の 3 つを確かめる。

1. **リークか**（式とデータの両面）：H を入れると、算出式を使って粘度を「計算」できてしまうか。H は、予測するときに事前に分かるか。
2. **H を外すとどうなるか**：Cross-WLF と同じ変数（γ̇、温度）だけで、見かけの粘度を予測できるか。温度に、設定温度 Tset を使う場合と、流動先端温度 Tff（流れの中の樹脂温度。Cross-WLF の「T」に近い）を使う場合。
3. **厚みの効果は何か**：同じせん断速度・同じ設定温度で、厚みによって見かけの粘度がどれだけ違うか。理想的なスリット流れ（等温・固化層なし）なら、見かけの流動曲線は厚みによらず重なる（理由は 2 節）。

**算出式**（スリット近似。L = 17.5 mm はセンサ間距離、W は幅、Δt は 2 点への到達の時間差、ΔP は圧力差）

| 量 | 式 | ln をとると |
|---|---|---|
| 流量 | Q = W·H·L/Δt | |
| 見かけのせん断速度 | γ̇ = 6Q/(W·H²) = 6L/(H·Δt) | ln γ̇ = −ln H − ln Δt + 定数 |
| 見かけの壁面せん断応力 | τ = ΔP·H/(2L) | ln τ = ln ΔP + ln H + 定数 |
| 見かけの粘度 | η = τ/γ̇ = ΔP·H²·Δt/(12L²) | **ln η = ln ΔP + ln H − ln γ̇ + 定数** |

ln η は、ΔP（流れの結果。測る値）、H（金型の寸法。事前に決まる）、γ̇（Δt と H から計算）で決まる。**ΔP（＝ P3_1 − 0.30 MPa）を入力に入れると、ln η を計算できてしまう（リーク。手順3a）**。H と γ̇ だけでは、ΔP が分からないので計算できない。

| 記号 | モデル（GPR。ARD Matern 5/2 ＋ 線形 ＋ ノイズ。γ̇ は条件平均） | 入力 | 位置づけ |
|---|---|---|---|
| R3 | 基本のモデル（手順3b） | γ̇, ln H, Tset | 今まで |
| RFo | R3 の H をフーリエ数に替える | γ̇, ln Fo, Tset | ln Fo = −(ln γ̇ + 3 ln H) ＋ 定数。H と同じ情報（構造 A・B が使う入力） |
| W0 | **Cross-WLF と同じ変数**（T ＝ 設定温度） | γ̇, Tset | H を外す |
| W1 | **Cross-WLF と同じ変数**（T ＝ 流動先端温度） | γ̇, ln Tff | H を外し、温度を、流れの中の樹脂温度にする |
| W1T | 参考 | γ̇, ln Tff, Tset | H を外し、2 つの温度 |

データは、手順3d の 2 つの版（外れ値なし。5data 版：S_V・S_T、Alldata 版：CV）。圧力（Cross-WLF の D3 項）は、粘度の結果なので、入力にしない（CLAUDE.md §5）。

## 使い方（このノートを一人で実行して、結果を判断するために）

1. メニューの「すべて実行」で、上から順に実行する（約 20 分）。カーネルは、このプロジェクトの `.venv` を選ぶ。
2. 設定を変えたいときは、最初の「設定エリア」のセルだけを直して、最初から実行し直す（高解像度のグラフの設定は、下の 7）。
3. 各節の最後に、**判定**が出る（`>> 判定：OK` か `>> 判定：★要確認`）。OK は「想定どおり」、★ は「想定と違う、または結果を読むときに注意が要る点」。基準は、判定の文の中に書いてある。★ が出ても、エラーではない。説明書の「判定の読み方」で、★ の意味を確かめる。
4. 図と表は `results/step3e/` に保存される。図の見方は、説明書 `results/学習結果説明_手順3e.pdf` にある。
5. データを取り直したとき、問題のあるデータを入れ替えたときは、`問題データ一覧.md` の手順に従う。このノートは、全条件のファイルを読み、`02_機械学習データ/問題データ一覧.csv` の「除外」を除く。
6. 用語や図の見方で分からないところは、説明書の用語集で調べる。記号（Tff、Tset、R3、E_h など）の意味は、`results/学習結果説明_記号表.pdf` に一覧がある。
7. **高解像度のグラフ**（報告書・発表資料用。GPR6月ノートのセル9-2 と同じ見せ方）は、「高解像度のグラフ」の節のセルで作る。**このセルだけは、セルの先頭にも設定エリアがあり、そこだけを直して、このセルだけを実行し直せる**（学習はやり直さない。全体を一度実行した後で）。保存先は `results/step3e/highres/`（Git には入れない）。

## 1. 設定

In [ ]:
# セル1：設定エリア  ▼▼▼ 変えてよいのは、ここだけ ▼▼▼
SEED = 42                     # 乱数シード
NOISE_LB = 1e-6               # ノイズ分散の下限（旧ノートは 1e-3）
N_RESTARTS = 5                # GPR の最適化のやり直し回数（増やすと遅いが、安定する）
USE_EXCLUSIONS = True         # True：問題データ一覧の「除外」を使う／False：全条件を使う（取り直した後の確認用）
REPO = ".."                   # リポジトリ直下（このノートは 01_機械学習コードフォルダ にある）
FIG_DPI = 200                 # 保存する PNG の解像度
DATA_VERSIONS = ["5data", "Alldata"]   # 検討する版（手順3d）。片方だけにもできる
MAIN_DATA = "なし"            # 外れ値なし（外れ値ありは使わない）
TEMP_HOLDOUT = 210            # S_T でテストにする設定温度 [℃]
CV_K = 10                     # Alldata 版の交差検証の分割数（条件単位）
N_RESTARTS_CV = 1             # 交差検証の各分割での最適化のやり直し回数
N_JOBS = 6                    # 並列に計算する数（CPU の数に合わせる）
KEYS = ["R3", "RFo", "W0", "W1", "W1T"]   # 比べるモデル（記号はノート冒頭の表）
REF_H = 2.0                   # 厚みの効果を比べる基準の厚み [mm]
HR_ALL = True                 # すべての図を、高解像度でも results/step3e/highres/ に保存するか（Git の管理外）
HR_ALL_DPI = 600              # その解像度
# ▲▲▲ 設定エリア終了 ▲▲▲

In [ ]:
# セル1.5：ライブラリ、フォント、出力先
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from joblib import Parallel, delayed

OUT_DIR = os.path.join(REPO, "results", "step3e")
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
try:
    fm.findfont("IPAexGothic", fallback_to_default=False)
except ValueError:
    raise RuntimeError("日本語フォント IPAexGothic が見つかりません。インストールしてからカーネルを再起動してください。")
plt.rcParams["font.family"] = ["IPAexGothic", "DejaVu Sans"]
print("Python:", sys.executable, "/", sys.version.split()[0])
print("出力先:", os.path.abspath(OUT_DIR))

### 共通の部品

データの読み込み、問題データ一覧の「除外」、モデル、評価、版ごとの分割（`version_sets`）、高解像度の図（`hr_flow`、`save_fig`）の関数。中身は `04_補助スクリプト/nb_common.py` と同じ。

In [ ]:
# セル：共通の部品（読み込み・モデル・評価）。04_補助スクリプト/nb_common.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, DotProduct, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- 設定（ノートの設定エリアの値があれば、それを使う）
SEED = globals().get("SEED", 42)
NOISE_LB = globals().get("NOISE_LB", 1e-6)                 # ノイズ分散の下限（CLAUDE.md §5。旧ノートは 1e-3）
N_RESTARTS = globals().get("N_RESTARTS", 5)                # GPR の最適化のやり直し回数
REPO = globals().get("REPO", os.environ.get("NB_REPO", ".."))   # リポジトリ直下（ノートは 01_機械学習コードフォルダ にあるので ".."。スクリプトは環境変数 NB_REPO で指定）
USE_EXCLUSIONS = globals().get("USE_EXCLUSIONS", True)     # 問題データ一覧の「除外」を使うか
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 下限への張り付きは kernels.csv に記録する

# ---------------------------------------------------------------- データの場所
DATA_DIR = os.path.join(REPO, "02_機械学習データ", "NR500")
TEST_DIR = os.path.join(REPO, "03_予測検証用テストデータ", "NR500")
PROBLEM_CSV = os.path.join(REPO, "02_機械学習データ", "問題データ一覧.csv")
MATS = {"PP": "PP", "MIX": "粉砕材混合PP"}
FILES = {   # (材料, 組): (上位フォルダ, 中間フォルダ, フォルダ, ファイル名)。どれも全条件のファイル（問題のある条件は、一覧で除く）
    ("PP", "5data"): (DATA_DIR, "02_バージンPP再々実験_5data", "03_pp_max_5data", "結合_03_pp_max_5data_外れ値{o}.csv"),
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "02_pp_max_3data", "結合_02_pp_max_3data_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "01_pp_max_Alldata", "結合_01_pp_max_Alldata_外れ値{o}.csv"),
    ("MIX", "5data"): (DATA_DIR, "04_粉砕材混合PP実験_5data", "03_mixpp_max_5data", "結合_03_mixpp_max_5data_外れ値{o}.csv"),
    ("MIX", "3data"): (TEST_DIR, "04_粉砕材混合PP実験_5data用検証データ(3data)", "02_mixpp_max_3data", "結合_02_mixpp_max_3data_外れ値{o}.csv"),
    ("MIX", "Alldata"): (DATA_DIR, "03_粉砕材混合PP実験_alldata", "01_mixpp_max_Alldata", "結合_01_mixpp_max_Alldata_外れ値{o}.csv"),
}
LEGACY_FILES = {   # 1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）。一覧の「除外」で同じデータになるかの確認用
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "05_pp_max_3data(no1.5-3.0)", "結合_05_pp_max_3data(no1.5-3.0)_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "04_pp_max_Alldata(no1.5-3.0)", "結合_04_pp_max_Alldata(no1.5-3.0)_外れ値{o}.csv"),
}
COLS = {"元ファイル": "file", "キャビ厚": "H", "射出速度[mm/s]": "v", "樹脂粘度η34(Pa・s)": "eta", "せん断速度γ34(1/s)": "gd",
        "流動中樹脂圧力P3_1(MPa)": "P31", "流動中樹脂圧力P3_2(MPa)": "P32", "加熱シリンダ温度[℃]": "Tset", "流動中樹脂温度(℃)": "Tff"}
RE_SRC = r"^(\d{3})_(\d+)℃_(\d{3})_([\d.]+)mm\.csv$"
D = {}                  # (材料, 組, 外れ値) → DataFrame。ノートで D = load_all() として入れる
REMOVED = {}            # (材料, 組, 外れ値) → 一覧の「除外」で除いた行数


def path_of(mat, grp, o, files=None):
    top, d1, d2, fn = (files or FILES)[(mat, grp)]
    return os.path.join(top, d1, d2, f"外れ値{o}", fn.format(o=o))


# ---------------------------------------------------------------- 問題データ一覧
def read_problem_list(path=None):
    """問題データ一覧.csv を読む（空欄は空文字）"""
    return pd.read_csv(path or PROBLEM_CSV, encoding="utf-8-sig", dtype=str).fillna("")


def problem_mask(df, mat, grp, plist, kinds):
    """区分が kinds のどれかで、再測定が「済み」でない一覧の行のうち、この材料・組・条件に当てはまる行の真偽値と、その ID を返す"""
    mask = np.zeros(len(df), bool)
    ids = [[] for _ in range(len(df))]
    for _, r in plist.iterrows():
        if r["区分"] not in kinds or r["再測定"] == "済み" or r["データ"] not in ("新", ""):
            continue
        if r["材料"] not in ("すべて", "", MATS[mat]) or r["組"] not in ("", grp):
            continue
        m = np.ones(len(df), bool)
        if r["設定温度"] != "":
            m &= df["Tset"].to_numpy() == float(r["設定温度"])
        if r["厚み"] != "":
            m &= np.isclose(df["H"].to_numpy(), float(r["厚み"]))
        if r["射出速度"] != "":
            m &= df["v"].to_numpy() == float(r["射出速度"])
        if r.get("ショット", "") != "":                      # 空欄は条件全体。数字なら、そのショット（元ファイル名の先頭 3 桁）だけ
            m &= (df["shot"].to_numpy() == int(r["ショット"])) if "shot" in df.columns else np.zeros(len(df), bool)
        mask |= m
        for i in np.flatnonzero(m):
            ids[i].append(r["ID"])
    return mask, [";".join(x) for x in ids]


def first_shots(mat, grp):
    """条件ごとの、最初のショットの番号（外れ値ありのファイル。ショットが除かれる前の番号）。ショットの順番 pos の基準に使う"""
    ref = pd.read_csv(path_of(mat, grp, "あり"), encoding="utf-8-sig").rename(columns=COLS)
    ref["shot0"] = ref["file"].str.extract(RE_SRC)[0].astype(int)
    ref["Tset"] = ref["Tset"].astype(float)
    return ref.groupby(["Tset", "H", "v"], as_index=False)["shot0"].min()


def load(mat, grp, o, plist=None, exclude=None):
    """結合ファイルを読み、形式を確かめ、問題データ一覧の「除外」を除く。列 sens には、「感度確認」の ID が入る（なければ空文字）。
    列 shot は元ファイル名の先頭 3 桁（設定温度・厚みごとの通し番号で、同じ条件のショットは連続した番号）。列 pos は、その条件の中での順番（最初のショット = 1。外れ値で除かれたショットの番号は欠ける）"""
    exclude = USE_EXCLUSIONS if exclude is None else exclude
    df = pd.read_csv(path_of(mat, grp, o), encoding="utf-8-sig")
    assert list(df.columns) == list(COLS), f"列名が想定と違う: {list(df.columns)}"
    df = df.rename(columns=COLS)
    assert df.isna().sum().sum() == 0, "欠損がある"
    p = df["file"].str.extract(RE_SRC)
    assert p.notna().all().all(), "ファイル名の形式に合わない行がある"
    assert np.allclose(p[1].astype(float), df["Tset"]) and np.allclose(p[2].astype(float), df["v"]) \
        and np.allclose(p[3].astype(float), df["H"]), "ファイル名と列の値が一致しない"
    df["shot"] = p[0].astype(int)
    df["Tset"] = df["Tset"].astype(float)
    df = df.merge(first_shots(mat, grp), on=["Tset", "H", "v"], how="left")
    assert df["shot0"].notna().all(), "外れ値ありのファイルにない条件がある"
    df["pos"] = df["shot"] - df["shot0"].astype(int) + 1
    df = df.drop(columns="shot0")
    plist = read_problem_list() if plist is None else plist
    n0 = len(df)
    if exclude:
        drop, _ = problem_mask(df, mat, grp, plist, ("除外",))
        df = df[~drop].reset_index(drop=True)
    REMOVED[(mat, grp, o)] = n0 - len(df)
    _, df["sens"] = problem_mask(df, mat, grp, plist, ("感度確認",))
    df["is_sens"] = df["sens"] != ""
    df["cond"] = df["Tset"].astype(int).astype(str) + "_" + df["H"].astype(str) + "_" + df["v"].astype(int).astype(str)
    df["ln_eta"], df["ln_gd"], df["ln_H"] = np.log(df["eta"]), np.log(df["gd"]), np.log(df["H"])
    df["ln_P31"], df["ln_P32"] = np.log(df["P31"] * 1e6), np.log(df["P32"] * 1e6)   # CSV の圧力は MPa。Pa に直してから ln をとる
    df["ln_v"] = np.log(df["v"])
    df["ln_Tff"] = np.log(df["Tff"] + 273.15)          # 流動先端温度 [K]
    for c in ["gd", "Tff", "P32"]:   # 条件ごとの平均（成形条件から見たばらつきを見るモデル用。cm は condition mean）
        df[f"ln_{c}_cm"] = df.groupby("cond")[f"ln_{c}"].transform("mean")
    return df


def load_all(exclude=None):
    """全材料・全組・外れ値なし/あり を読む。学習と検証で条件が重ならないことも確かめる"""
    plist = read_problem_list()
    out = {(mat, grp, o): load(mat, grp, o, plist, exclude) for mat in MATS for grp in ["5data", "3data", "Alldata"] for o in ["なし", "あり"]}
    for mat in MATS:
        for o in ["なし", "あり"]:
            assert not (set(out[(mat, "5data", o)]["cond"]) & set(out[(mat, "3data", o)]["cond"])), "学習と検証で条件が重なっている"
    return out


def legacy_check(D_):
    """全条件のファイルから一覧で除いたデータが、1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）と同じ行か（PP の 3data・Alldata）。
    no1.5-3.0 のファイルにも、1.5 mm・3 mm/s 以外の一覧の「除外」（P03 など）を同じように適用してから比べる"""
    rows = []
    plist = read_problem_list()
    plist = plist[~((plist["厚み"] == "1.5") & (plist["射出速度"] == "3") & (plist["設定温度"] == ""))]      # no1.5-3.0 のファイルで、すでに除かれている分
    for (mat, grp), _ in LEGACY_FILES.items():
        for o in ["なし", "あり"]:
            old = pd.read_csv(path_of(mat, grp, o, LEGACY_FILES), encoding="utf-8-sig")
            o2 = old.rename(columns=COLS)
            o2["shot"] = o2["file"].str.extract(RE_SRC)[0].astype(int)
            o2["Tset"] = o2["Tset"].astype(float)
            drop, _ = problem_mask(o2, mat, grp, plist, ("除外",)) if USE_EXCLUSIONS else (np.zeros(len(o2), bool), None)
            old = old[~drop]
            same = set(old["元ファイル"]) == set(D_[(mat, grp, o)]["file"])
            rows.append({"材料": MATS[mat], "組": grp, "外れ値": o, "一覧で除いた後の行数": len(D_[(mat, grp, o)]), "no1.5-3.0 の行数": len(old), "同じ行": same})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- モデル
def kernel_for(d, linear=False):
    """ARD Matern 5/2 ＋ ノイズ。linear=True なら、線形カーネル（DotProduct）も足す（低速側への外挿のため。CLAUDE.md §5）"""
    k = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    if linear:
        k = k + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))
    return k + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


def make_gpr(d, linear=False, restarts=None):
    return GaussianProcessRegressor(kernel=kernel_for(d, linear), n_restarts_optimizer=N_RESTARTS if restarts is None else restarts, random_state=SEED)


def fit_ols(X_tr, y_tr):
    """切片ありの最小二乗回帰。係数、(XᵀX)⁻¹、残差分散を返す"""
    A_tr = np.column_stack([np.ones(len(X_tr)), X_tr])
    beta, *_ = np.linalg.lstsq(A_tr, y_tr, rcond=None)
    resid = y_tr - A_tr @ beta
    return beta, np.linalg.inv(A_tr.T @ A_tr), resid @ resid / (len(y_tr) - A_tr.shape[1])


def predict_ols(model, X):
    """予測平均と、新しい1ショットに対する予測標準偏差"""
    beta, XtX_inv, s2 = model
    A_new = np.column_stack([np.ones(len(X)), X])
    lev = np.einsum("ij,jk,ik->i", A_new, XtX_inv, A_new)
    return A_new @ beta, np.sqrt(s2 * (1.0 + lev))


MODELS = {}             # 記号 → {"kind": "ols" か "gpr", "cols": [入力の列], "linear": 線形カーネルを足すか}。ノートで決める


def fit_model(key, tr, restarts=None):
    spec = MODELS[key]
    X, y = tr[spec["cols"]].to_numpy(), tr["ln_eta"].to_numpy()
    if spec["kind"] == "ols":
        return {"kind": "ols", "model": fit_ols(X, y)}
    sc = StandardScaler().fit(X)              # 標準化と中心化は学習データだけで計算する
    y0 = y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False), restarts).fit(sc.transform(X), y - y0)
    return {"kind": "gpr", "gp": gp, "sc": sc, "y0": y0}


def predict_model(key, fit, te):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X = te[MODELS[key]["cols"]].to_numpy()
    if fit["kind"] == "ols":
        mu, sd = predict_ols(fit["model"], X)
        return mu, sd, np.full(len(X), np.nan), float(np.sqrt(fit["model"][2]))
    mu, sd = fit["gp"].predict(fit["sc"].transform(X), return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"], sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def evaluate(y, mu, sd):
    """ln η の指標：R2、RMSE、95% 予測区間の被覆率、NLPD（各ショットの予測分布の負の対数密度の平均）、予測標準偏差の平均"""
    z = (y - mu) / sd
    return {"R2": r2_score(y, mu) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean(np.abs(z) <= 1.96)),
            "NLPD": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + 0.5 * z ** 2)), "mean_sd": float(np.mean(sd))}


def kernel_row(key, fit):
    """学習した GPR のハイパーパラメータ（振幅、長さスケール、ノイズ、線形項、対数周辺尤度、下限・上限への張り付き）"""
    gp = fit["gp"]
    k = gp.kernel_
    lin = MODELS[key].get("linear", False)
    prod, lin_prod = (k.k1.k1, k.k1.k2) if lin else (k.k1, None)
    ls = np.atleast_1d(prod.k2.length_scale)
    row = {"amplitude": float(np.sqrt(prod.k1.constant_value)), "noise_var": float(k.k2.noise_level),
           "noise_sd": float(np.sqrt(k.k2.noise_level)), "LML": float(gp.log_marginal_likelihood_value_),
           "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05),
           "ls_at_bound": bool(np.any(ls > 99) or np.any(ls < 0.0105))}
    if lin_prod is not None:
        row.update({"lin_amplitude": float(np.sqrt(lin_prod.k1.constant_value)), "lin_sigma0": float(lin_prod.k2.sigma_0)})
    row.update({f"ls_{c}": float(l) for c, l in zip(MODELS[key]["cols"], ls)})
    return row


def within_stats(df):
    """同じ条件内のばらつき（プール）。dof：自由度、var_after_gd：ln γ̇ で説明した残りの分散"""
    g = df.groupby("cond")
    dof = int((g.size() - 1).sum())
    de = df["ln_eta"] - g["ln_eta"].transform("mean")
    dg = df["ln_gd"] - g["ln_gd"].transform("mean")
    var_e = float((de ** 2).sum() / dof)
    b = float((de * dg).sum() / (dg ** 2).sum())
    return {"dof": dof, "var_ln_eta": var_e, "sd_ln_eta": var_e ** 0.5, "corr_with_ln_gd": float((de * dg).sum() / np.sqrt((de ** 2).sum() * (dg ** 2).sum())),
            "slope_vs_ln_gd": b, "var_after_gd": float(((de - b * dg) ** 2).sum() / (dof - 1))}


# ---------------------------------------------------------------- データの分け方
def split_V(mat, o):
    """5data で学習し、3data で検証する（射出速度の両端の外挿）。te の group 列は、低速側（3・5 mm/s）か高速側（280・320 mm/s）"""
    tr, te = D[(mat, "5data", o)].copy(), D[(mat, "3data", o)].copy()
    te["group"] = np.where(te["v"] <= 5, "低速側", np.where(te["v"] >= 280, "高速側", "その他"))
    assert (te["group"] != "その他").all(), "検証データに、低速側・高速側のどちらでもない条件がある"
    return tr, te


def split_random(df):
    """行単位のランダム 20%（従来の分け方。random_state = SEED）"""
    idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
    te = df.iloc[idx_te].copy()
    te["group"] = "全体"
    return df.iloc[idx_tr].copy(), te


def split_T(mat, o, t_test=210):
    """設定温度 t_test の全条件をテストにし、5data の残りの温度で学習する（温度の内挿の検証）。3data の同じ温度は te3 として別に返す"""
    d5, d3 = D[(mat, "5data", o)], D[(mat, "3data", o)]
    tr, te = d5[d5["Tset"] != t_test].copy(), d5[d5["Tset"] == t_test].copy()
    te["group"] = "全体"
    te3 = d3[d3["Tset"] == t_test].copy()
    te3["group"] = np.where(te3["v"] <= 5, "低速側", "高速側")
    return tr, te, te3


def pool_of(mat, o):
    """5data と 3data をまとめたデータ"""
    return pd.concat([D[(mat, "5data", o)], D[(mat, "3data", o)]], ignore_index=True)


# ---------------------------------------------------------------- 学習と評価の記録
METRICS, KERNELS, OLSROWS, PRED, FITS = [], [], [], {}, {}      # FITS：(材料, split, データ, モデル) → 学習した予測器
SENS_SUFFIX = "（感度確認の条件を除く）"


def _record(mat, name, data, key, fit, tr, te, pure):
    """学習した予測器 fit で te を予測し、指標（subset ごと）と予測を記録する"""
    subsets = {"全体": np.ones(len(te), bool)}
    for g in te["group"].unique():
        if g != "全体":
            subsets[g] = (te["group"] == g).to_numpy()
    if te["is_sens"].any():
        subsets.update({s + SENS_SUFFIX: m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
    y_te = te["ln_eta"].to_numpy()
    mu, sd, sd_f, noise_sd = predict_model(key, fit, te)
    PRED[(mat, name, data, key)] = pd.DataFrame({"y": y_te, "mu": mu, "sd": sd, "sd_f": sd_f, "v": te["v"].to_numpy(), "H": te["H"].to_numpy(),
                                                 "Tset": te["Tset"].to_numpy(), "cond": te["cond"].to_numpy(), "group": te["group"].to_numpy(),
                                                 "is_sens": te["is_sens"].to_numpy()})
    for sub, mask in subsets.items():
        if mask.sum() == 0:
            continue
        ev = evaluate(y_te[mask], mu[mask], sd[mask])
        METRICS.append({"material": MATS[mat], "split": name, "data": data, "model": key, "subset": sub, "n_train": len(tr),
                        "n_test": int(mask.sum()), "n_cond_test": int(te.loc[mask, "cond"].nunique()), **ev,
                        "mean_sd_latent": float(np.nanmean(sd_f[mask])) if fit["kind"] == "gpr" else np.nan, "noise_sd": noise_sd})


def run_split(mat, split, data, tr, te, keys=None, tag="", extra_te=None):
    """全モデルを学習し、指標（subset ごと）・カーネル・予測を記録する。te には group 列がある。
    te に「感度確認」の条件があれば、各 subset から除いた指標（subset 名の後ろに SENS_SUFFIX）も記録する。
    tag は split 名の後ろに付ける印。extra_te（{split 名: テスト}）があれば、同じ学習で、それらも評価する"""
    keys = keys or ALL_KEYS
    name = split + tag
    tests = {name: te, **{s + tag: t for s, t in (extra_te or {}).items()}}
    pure = within_stats(tr)
    for key in keys:
        t0 = time.time()
        fit = fit_model(key, tr)
        FITS[(mat, name, data, key)] = fit
        for nm, te_ in tests.items():
            _record(mat, nm, data, key, fit, tr, te_, pure)
        base = {"material": MATS[mat], "split": name, "data": data, "model": key, "n_train": len(tr)}
        if fit["kind"] == "gpr":
            KERNELS.append({**base, **kernel_row(key, fit), "pure_sd_ln_eta": pure["sd_ln_eta"], "pure_sd_after_gd": pure["var_after_gd"] ** 0.5})
        else:
            names = ["intercept"] + MODELS[key]["cols"]
            OLSROWS.append({**base, "resid_sd": float(np.sqrt(fit["model"][2])), **dict(zip(names, map(float, fit["model"][0])))})
        print(f"{MATS[mat]} {name} {data} {key}: 学習 {len(tr)} 行、{time.time() - t0:.1f} 秒")


# ---------------------------------------------------------------- 条件を1つずつ抜く交差検証（LOCO）
def loco_gpr(key, pool):
    """GPR：ハイパーパラメータを全データで決めて固定し、条件を抜いた予測を閉じた式で計算する（平均 = y_g − [K⁻¹_gg]⁻¹(K⁻¹y)_g、共分散 = [K⁻¹_gg]⁻¹）"""
    spec = MODELS[key]
    X, y = pool[spec["cols"]].to_numpy(), pool["ln_eta"].to_numpy()
    sc = StandardScaler().fit(X)
    Xs, yc = sc.transform(X), y - y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False)).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))              # ノイズ込みの共分散行列の逆行列
    alpha = Kinv @ yc
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        Minv = np.linalg.inv(Kinv[np.ix_(idx, idx)])
        mu[idx] = y[idx] - Minv @ alpha[idx]
        sd[idx] = np.sqrt(np.diag(Minv))
    return mu, sd, gp


def loco_ols(key, pool):
    """線形回帰：条件ごとに、その条件を除いて当てはめ直す"""
    cols = MODELS[key]["cols"]
    X, y = pool[cols].to_numpy(), pool["ln_eta"].to_numpy()
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        keep = np.ones(len(y), bool)
        keep[idx] = False
        mu[idx], sd[idx] = predict_ols(fit_ols(X[keep], y[keep]), X[idx])
    return mu, sd


# ---------------------------------------------------------------- 外れ値の検知（規則 f）と、データの版（5data 版・Alldata 版）
OUTLIER_Z = globals().get("OUTLIER_Z", 3.5)             # 規則 f：|z| がこれを超え、かつ |残差|（ln η）が次の値を超える条件を候補にする
OUTLIER_RESID = globals().get("OUTLIER_RESID", 0.1)     # 0.1 は粘度で約 10%


def cond_means(df):
    """条件ごとの平均（ln η と ln γ̇ をショットで平均した点）"""
    g = df.groupby("cond", sort=False)
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"), n=("ln_eta", "size")).reset_index()
    t["ln_H"] = np.log(t["H"])
    return t


def outlier_scan(df, z_thr=None, resid_thr=None):
    """規則 f：条件平均が、ほかの条件から予測した値と大きく違う条件を拾う。
    条件平均の GPR（入力 ln γ̇（条件平均）・ln H・Tset。R3 と同じ形）の、条件を 1 つずつ抜いた予測（閉じた式）との残差を、予測標準偏差で割って z とする。
    返す表の列：cond, Tset, H, v, n（ショット数）, resid（ln η の実測 − 予測）, sd, z, flag（|z| と |resid| が両方しきい値を超えたか）"""
    z_thr = OUTLIER_Z if z_thr is None else z_thr
    resid_thr = OUTLIER_RESID if resid_thr is None else resid_thr
    t = cond_means(df)
    X, y = t[["ln_gd", "ln_H", "Tset"]].to_numpy(), t["ln_eta"].to_numpy()
    Xs, yc = StandardScaler().fit_transform(X), y - y.mean()
    gp = make_gpr(3, True).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))
    d = np.diag(Kinv)
    t["resid"], t["sd"] = (Kinv @ yc) / d, np.sqrt(1.0 / d)           # 条件 i を抜いた予測：平均 = y_i − (K⁻¹y)_i / K⁻¹_ii、分散 = 1 / K⁻¹_ii
    t["z"] = t["resid"] / t["sd"]
    t["flag"] = (t["z"].abs() > z_thr) & (t["resid"].abs() > resid_thr)
    return t[["cond", "Tset", "H", "v", "n", "resid", "sd", "z", "flag"]]


DATA_VERSIONS = globals().get("DATA_VERSIONS", ["5data", "Alldata"])   # 検討するデータの版。"5data"：5data で学習し、3data で外挿を検証する版／"Alldata"：全条件で学習する版
CV_K = globals().get("CV_K", 10)                                       # Alldata 版の交差検証の分割数（条件単位）


def split_cv(pool, k=None):
    """条件単位の k 分割交差検証。同じ条件のショットは同じ側に入れる（SEED で条件を並べ替える）。(学習, テスト) の組の一覧を返す"""
    k = CV_K if k is None else k
    conds = np.array(sorted(pool["cond"].unique()))
    np.random.RandomState(SEED).shuffle(conds)
    out = []
    for i in range(k):
        te_c = set(conds[i::k])
        te = pool[pool["cond"].isin(te_c)].copy()
        te["group"] = "全体"
        out.append((pool[~pool["cond"].isin(te_c)].copy(), te))
    return out


def version_sets(mat, version, data="なし", t_test=210):
    """データの版ごとの分割を返す。{"pool": 最終モデル（ALL）の学習データ, "splits": {分割の名前: (学習, {集計の名前: テスト})}}
    5data 版：S_V（5data → 3data。低速側・高速側）、S_T（190・230 ℃ → 210 ℃。5data の 210 ℃ と 3data の 210 ℃ を別に集計）、CV（5data の中の条件単位の CV_K 分割）。最終モデルは 5data（3data は検証専用）。
    Alldata 版：S_T（Alldata の 190・230 ℃ で学習。テストは 5data 版と同じ行：5data ファイルの 210 ℃ と 3data ファイルの 210 ℃。そのまま比べられる）、
    S_Vall（Alldata のうち 5data の条件だけで学習 → 3data の条件を検証。PP のみ意味がある〔粉砕材混合 PP は 5data と同じ〕）、CV（Alldata の条件単位の CV_K 分割）。最終モデルは Alldata（全条件）"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if version == "5data":
        tr_v, te_v = split_V(mat, data)
        tr_t, te_t, te_t3 = split_T(mat, data, t_test)
        sp = {"S_V": (tr_v, {"S_V": te_v}), "S_T": (tr_t, {"S_T": te_t, "S_T3": te_t3})}
        pool = d5
    else:
        assert version == "Alldata", version
        pool = D[(mat, "Alldata", data)]
        _, te_t, te_t3 = split_T(mat, data, t_test)
        sp = {"S_T": (pool[pool["Tset"] != t_test].copy(), {"S_T": te_t, "S_T3": te_t3})}
        te_v = d3.copy()
        te_v["group"] = np.where(te_v["v"] <= 5, "低速側", np.where(te_v["v"] >= 280, "高速側", "その他"))
        sp["S_Vall"] = (pool[pool["cond"].isin(set(d5["cond"]))].copy(), {"S_Vall": te_v})
    for i, (tr_i, te_i) in enumerate(split_cv(pool), 1):
        sp[f"CV{i}"] = (tr_i, {f"CV{i}": te_i})
    return {"pool": pool, "splits": sp}


# ---------------------------------------------------------------- 判定の表示（GPR6月ノートの「判定」にならう）
def judge(ok, text):
    """結果の判定を1行で表示する。ok が True なら「OK」、False なら「要確認」"""
    print(f">> 判定：{'OK' if ok else '★要確認'}　{text}")


# ---------------------------------------------------------------- 高解像度のグラフ（GPR6月ノートのセル9-2 にならう。報告書・発表資料用）
HR_COLORS = {"H": {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"},       # 厚みごとの色（6月のノートと同じ並び。黄は、線で見やすい濃さにした）
             "Tset": {190: "#1f77b4", 210: "#2ca02c", 230: "#d62728"}}                       # 設定温度ごとの色
HR_MARKS = {"H": {0.5: "o", 1.0: "s", 1.5: "^", 2.0: "D"}, "Tset": {190: "o", 210: "s", 230: "^"}}
HR_DEFAULT = dict(layouts=["thickness", "temperature"], band="predict", points="cond", show_test=True, mark_sens=True, x_range=None, extend=1.3,
                  xlim={"thickness": None, "temperature": (30, 50000)}, ylim={"thickness": None, "temperature": (0.5, 5000)}, figsize=(15, 10), dpi=600, font={"axis": 24, "legend": 24, "title": 24, "tick": 28},
                  marker=200, marker_shot=70, line=3.0, show_title=True, show_xlabel=False, show_ylabel=False, formats=["png"], show=False, n_grid=120)


def hr_split_conds(mat, data, split, t_test=210):
    """学習に使った条件の集合。S_V：5data／S_T：5data の t_test 以外の温度／ALL：5data と 3data"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if split == "S_V":
        return set(d5["cond"])
    if split == "S_T":
        return set(d5.loc[d5["Tset"] != t_test, "cond"])
    return set(d5["cond"]) | set(d3["cond"])


def hr_inputs(sub, lg, h, t):
    """予測線の入力（ln γ̇ = lg の点）。Tff・P3_2・射出速度は、同じ設定温度・厚みの条件平均を ln γ̇ で内挿する（どの入力のモデルでも描けるように）"""
    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), tff=("ln_Tff", "mean"), p32=("ln_P32", "mean"), lv=("ln_v", "mean")).sort_values("lg")
    ip = lambda c: np.interp(lg, cm["lg"], cm[c])
    return pd.DataFrame({"ln_gd": lg, "ln_gd_cm": lg, "ln_H": np.log(h), "Tset": float(t), "ln_Tff": ip("tff"), "ln_Tff_cm": ip("tff"),
                         "ln_P32": ip("p32"), "ln_P32_cm": ip("p32"), "ln_v": ip("lv")})


HR_ALL = globals().get("HR_ALL", True)                 # すべての図を、高解像度でも保存するか（True：out_dir/highres/ に HR_ALL_DPI で保存。Git の管理外）
HR_ALL_DPI = globals().get("HR_ALL_DPI", 600)


def save_fig(fig, out_dir, stem):
    """図を out_dir に PNG で保存する（解像度は FIG_DPI）。HR_ALL が True なら、同じ図を HR_ALL_DPI（既定 600 dpi）で out_dir/highres/ にも保存する。保存したファイルの一覧を返す"""
    os.makedirs(out_dir, exist_ok=True)
    paths = [os.path.join(out_dir, f"{stem}.png")]
    fig.savefig(paths[0], dpi=globals().get("FIG_DPI", 200), bbox_inches="tight")
    if HR_ALL:
        os.makedirs(os.path.join(out_dir, "highres"), exist_ok=True)
        paths.append(os.path.join(out_dir, "highres", f"{stem}.png"))
        fig.savefig(paths[1], dpi=HR_ALL_DPI, bbox_inches="tight")
    return paths


def hr_flow(predict, mat, data, train_conds, title_model, stem, out_dir, pool=None, **kw):
    """せん断速度 ― 粘度の学習結果グラフを、高解像度で 1 枚ずつ保存する（6月のノートのセル9-2 と同じ見せ方）。
    predict(Xg) → (予測平均 ln η, 予測標準偏差（ノイズ込み）, 潜在関数の標準偏差)。Xg は hr_inputs の DataFrame。
    train_conds：学習に使った条件の集合（それ以外は、検証の点として白抜きで重ねる）。pool：点と曲線の範囲に使うデータ（None なら 5data＋3data）。
    設定は HR_DEFAULT を参照（kw で上書き）。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = {**HR_DEFAULT, **kw}
    os.makedirs(out_dir, exist_ok=True)
    pool = pool_of(mat, data) if pool is None else pool
    curves = {}                                                         # (厚み, 設定温度) → (x, 予測平均, 帯の幅の標準偏差)。レイアウトが違っても同じ線なので、1 回だけ計算する
    for (h, t), sub in pool.groupby(["H", "Tset"]):
        lgc = sub.groupby("cond")["ln_gd"].mean()
        if len(lgc) < 2:
            continue
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        lg = np.linspace(lo, hi, cfg["n_grid"])
        mu, sd, sdf = predict(hr_inputs(sub, lg, h, t))
        curves[(float(h), int(t))] = (np.exp(lg), np.asarray(mu), np.asarray(sd if cfg["band"] == "predict" else sdf))
    saved = []
    ft = cfg["font"]
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    for layout in cfg["layouts"]:
        by, sk = ("H", "Tset") if layout == "thickness" else ("Tset", "H")        # by：1 枚ごとに固定する量、sk：1 枚の中で線を分ける量
        for gv in sorted({k[0] if by == "H" else k[1] for k in curves}):
            fig, ax = plt.subplots(figsize=cfg["figsize"], dpi=cfg["dpi"])
            series, any_test, any_sens, xs, ys = [], False, False, [], []
            for sv in sorted({(k[1] if by == "H" else k[0]) for k in curves if (k[0] if by == "H" else k[1]) == gv}):
                h, t = (gv, sv) if by == "H" else (sv, gv)
                col, mk = HR_COLORS[sk][sv], HR_MARKS[sk][sv]
                x, mu, s = curves[(float(h), int(t))]
                if cfg["band"] in ("predict", "latent"):
                    ax.fill_between(x, np.exp(mu - 1.96 * s), np.exp(mu + 1.96 * s), color=col, alpha=0.22, zorder=15)
                    ax.plot(x, np.exp(mu - 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                    ax.plot(x, np.exp(mu + 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                ax.plot(x, np.exp(mu), color=col, lw=cfg["line"], zorder=20)
                sub = pool[(np.isclose(pool["H"], h)) & (pool["Tset"] == t)]
                if cfg["points"] == "cond":
                    a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    px, py, tr_m, se_m, msize = np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), a["cond"].isin(train_conds).to_numpy(), a["sens"].to_numpy().astype(bool), cfg["marker"]
                else:
                    px, py, tr_m, se_m, msize = sub["gd"].to_numpy(), sub["eta"].to_numpy(), sub["cond"].isin(train_conds).to_numpy(), sub["is_sens"].to_numpy().astype(bool), cfg["marker_shot"]
                xs += list(x) + list(px)
                ys += list(np.exp(mu)) + list(py)
                ax.scatter(px[tr_m], py[tr_m], s=msize, color=col, marker=mk, alpha=0.85, edgecolors="k", linewidths=0.6, zorder=14 - list(HR_COLORS[sk]).index(sv))
                if cfg["show_test"] and (~tr_m).any():
                    ax.scatter(px[~tr_m], py[~tr_m], s=msize, facecolors="white", edgecolors=col, marker=mk, linewidths=2.5, zorder=14 - list(HR_COLORS[sk]).index(sv))
                    any_test = True
                if cfg["mark_sens"] and se_m.any():
                    ax.scatter(px[se_m], py[se_m], s=msize * 1.3, color="k", marker="x", linewidths=3, zorder=25)
                    any_sens = True
                series.append((sv, col, mk))
            head = "キャビティ厚み" if sk == "H" else "設定温度"
            handles = [Patch(facecolor="gray", alpha=0.3, label=band_name), Patch(color="none", label=head)]
            handles += [Line2D([], [], color=col, lw=cfg["line"], marker=mk, markersize=np.sqrt(cfg["marker"]), markeredgecolor="k", markeredgewidth=0.6,
                               label=(f"{sv} mm" if sk == "H" else f"{int(sv)} ℃")) for sv, col, mk in series]
            if any_test:
                handles.append(Line2D([], [], color="gray", ls="none", marker="o", markersize=np.sqrt(cfg["marker"]), markerfacecolor="white", markeredgewidth=2.5, label="検証の条件（白抜き）"))
            if any_sens:
                handles.append(Line2D([], [], color="k", ls="none", marker="x", markersize=np.sqrt(cfg["marker"]), markeredgewidth=3, label="感度確認の条件"))
            if cfg["band"] not in ("predict", "latent"):
                handles = handles[1:]
            ax.legend(handles=handles, loc="upper right", fontsize=ft["legend"], framealpha=1.0, edgecolor="black")
            ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
            ax.set_xscale("log")
            ax.set_yscale("log")
            xl = cfg["xlim"][layout] if isinstance(cfg["xlim"], dict) else cfg["xlim"]       # 範囲：(下限, 上限)＝固定／None＝その図のデータに合わせる
            yl = cfg["ylim"][layout] if isinstance(cfg["ylim"], dict) else cfg["ylim"]
            auto_x, auto_y = xl is None, yl is None
            ax.set_xlim(*((min(xs) / 1.4, max(xs) * 1.4) if auto_x else xl))
            ax.set_ylim(*((min(ys) / 1.5, max(ys) * 1.5) if auto_y else yl))
            for axis, auto in ((ax.xaxis, auto_x), (ax.yaxis, auto_y)):
                if auto:                               # 範囲が狭いので、1・2・5 × 10^n に目盛りを付けて、普通の数で表す
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
                else:                                  # 6月のノートと同じ（10 の累乗の目盛り）
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
                    axis.set_major_formatter(ticker.LogFormatterSciNotation())
                axis.set_minor_formatter(ticker.NullFormatter())
            ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
            ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
            gtxt = f"厚み {gv} mm" if by == "H" else f"設定温度 {int(gv)} ℃"
            if cfg["show_title"]:
                ax.set_title(f"{MATS[mat]}　せん断速度と粘度の関係（{gtxt}）\n{title_model}", fontsize=ft["title"], pad=15)
            if cfg["show_xlabel"]:
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], labelpad=15)
            if cfg["show_ylabel"]:
                ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=ft["axis"], labelpad=15)
            fig.tight_layout()
            base = f"{stem}_{'by-thickness' if by == 'H' else 'by-temperature'}_{(str(gv) + 'mm') if by == 'H' else (str(int(gv)) + 'C')}"
            for fmt in cfg["formats"]:
                path = os.path.join(out_dir, f"{base}.{fmt}")
                fig.savefig(path, bbox_inches="tight", dpi=cfg["dpi"])
                saved.append(path)
            if cfg["show"]:
                plt.show()
            plt.close(fig)
    return saved

## 2. データの読み込み

全条件のファイルを読み、問題データ一覧の「除外」を除く。フーリエ数の列 `ln_Fo_cm` ＝ −(ln γ̇（条件平均）＋ 3 ln H) を足す（定数は GP に影響しないので省く）。

**理想的なスリット流れ**（等温で、固化層も壁面すべりもない）では、見かけの粘度と見かけのせん断速度の関係（見かけの流動曲線）は、**厚みによらず 1 本に重なる**。見かけの値と真の値の換算（ラビノビッチの補正）は、流動曲線の傾きだけで決まり、厚みに関係しないため（スリットやキャピラリーの寸法を変えて、流動曲線が重なるかで、壁面すべりなどを調べる方法〔Mooney の方法〕の前提）。**厚みで曲線がずれるなら、そのずれは、計算の都合ではなく、流れの中で起きている現象（冷却・固化層・せん断発熱など）による。**

In [ ]:
# セル2：読み込み
D = load_all()
for k in D:
    D[k]["ln_Fo_cm"] = -(D[k]["ln_gd_cm"] + 3.0 * D[k]["ln_H"])
leg = legacy_check(D)
judge(bool(leg["同じ行"].all()) or not USE_EXCLUSIONS, "一覧の「除外」を除いたデータが、no1.5-3.0 のファイル（P03 などの除外を同じように適用したもの）と同じ")
for mat in MATS:
    for g in ["5data", "3data", "Alldata"]:
        d = D[(mat, g, MAIN_DATA)]
        print(f"{MATS[mat]} {g}：{len(d)} 行 {d['cond'].nunique()} 条件")

## 3. 式の確認：算出式に入っている量だけで、ln η を再現できるか

ln η を、算出式に入っている量で、線形回帰する（ショットごと。5data で学習 → 3data で検証）。**リークなら、検証の RMSE が、同じ条件のショット間のばらつき（純粋誤差、約 0.04）よりずっと小さくなる**（計算で答えが出るため）。

| 回帰の入力 | 予想 |
|---|---|
| ln γ̇, ln H | H が式の一部としてだけ効くなら、ln H の係数は ＋1。実際は ΔP が分からないので、再現できないはず |
| ln γ̇, ln H, Tset | 同上 |
| ln Δt, ln H | ln γ̇ と同じ情報（γ̇ = 6L/(H·Δt)）。同じ結果になるはず |
| ln γ̇, ln H, ln ΔP | **算出式そのもの**。係数は (−1, ＋1, ＋1) で、完全に再現できるはず（リーク） |
| ln γ̇, ln ΔP | H なしでも、ΔP があれば大部分を再現できるか |

In [ ]:
# セル3：算出式の量での線形回帰
FORMS = {"ln γ̇ + ln H": ["ln_gd", "ln_H"], "ln γ̇ + ln H + Tset": ["ln_gd", "ln_H", "Tset"], "ln Δt + ln H": ["ln_dt", "ln_H"],
         "ln γ̇ + ln H + ln ΔP（算出式）": ["ln_gd", "ln_H", "ln_dP"], "ln γ̇ + ln ΔP": ["ln_gd", "ln_dP"]}
frows = []
for mat in MATS:
    tr, te = split_V(mat, MAIN_DATA)
    for d in (tr, te):
        d["ln_dP"] = np.log((d["P31"] - 0.30) * 1e6)                          # ΔP [Pa]（P3_1 − 0.30 MPa）
        d["ln_dt"] = np.log(6 * 17.5e-3 / (d["H"] * 1e-3 * d["gd"]))          # Δt [s]（γ̇ = 6L/(H·Δt) から）
    pure = within_stats(tr)["sd_ln_eta"]
    for name, cols in FORMS.items():
        m = fit_ols(tr[cols].to_numpy(), tr["ln_eta"].to_numpy())
        mu, _ = predict_ols(m, te[cols].to_numpy())
        rm = float(np.sqrt(np.mean((te["ln_eta"].to_numpy() - mu) ** 2)))
        frows.append({"material": MATS[mat], "inputs": name, **{f"coef_{c}": float(b) for c, b in zip(cols, m[0][1:])}, "resid_sd_train": float(np.sqrt(m[2])),
                      "RMSE_test": rm, "pure_sd": pure, "RMSE_over_pure": rm / pure})
form_df = pd.DataFrame(frows)
form_df.to_csv(os.path.join(OUT_DIR, "formula_check.csv"), index=False, encoding="utf-8-sig")
print(form_df.round(4).to_string(index=False))
for mat in MATS:
    s = form_df[form_df["material"] == MATS[mat]].set_index("inputs")
    a, b = s.loc["ln γ̇ + ln H", "RMSE_over_pure"], s.loc["ln γ̇ + ln H + ln ΔP（算出式）", "RMSE_over_pure"]
    judge(a > 2, f"{MATS[mat]}：ln γ̇ と ln H だけの回帰の検証 RMSE は、純粋誤差の {a:.1f} 倍（2 倍を超えれば、H と γ̇ からは ln η を計算できない ＝ 式によるリークではない）。ln H の係数は {s.loc['ln γ̇ + ln H', 'coef_ln_H']:+.2f}（式の一部としてだけ効くなら ＋1）")
    judge(b < 0.1, f"{MATS[mat]}：ΔP（P3_1 − 0.30 MPa）を足すと、検証 RMSE は純粋誤差の {b:.4f} 倍（ほぼ 0 ＝ ΔP はリーク。手順3a と同じ）")

## 4. 厚みの効果：同じ γ̇・同じ設定温度で、厚みによって粘度がどれだけ違うか

条件平均の見かけの流動曲線（ln γ̇ と ln η）を、厚みごとに、ln γ̇ について直線で補間し、**基準の厚み（`REF_H`）と γ̇ が重なる範囲で、粘度の比 η(H)/η(基準)** を出す。比べるために、**同じ厚み・同じ γ̇ での 190/230 ℃ の粘度の比**（温度の効果。手順3b・5a で 1.07〜1.21 倍）も同じ方法で出す。理想的なスリット流れなら、厚みの比は 1（曲線が重なる）。

In [ ]:
# セル4：厚みの比と、温度の比
def curve_ratio(cm, key_col, a_val, b_val, by_col):
    """by_col の値ごとに、key_col が a_val の曲線と b_val の曲線を、ln γ̇ が重なる範囲で比べる（ln η の差の平均・最小・最大）"""
    rows = []
    for bv, g in cm.groupby(by_col):
        A = g[np.isclose(g[key_col], a_val)].sort_values("ln_gd")
        B = g[np.isclose(g[key_col], b_val)].sort_values("ln_gd")
        if len(A) < 2 or len(B) < 2:
            continue
        lo, hi = max(A["ln_gd"].min(), B["ln_gd"].min()), min(A["ln_gd"].max(), B["ln_gd"].max())
        if hi <= lo:
            continue
        x = np.linspace(lo, hi, 50)
        d = np.interp(x, A["ln_gd"], A["ln_eta"]) - np.interp(x, B["ln_gd"], B["ln_eta"])
        rows.append({by_col: bv, "gd_lo": float(np.exp(lo)), "gd_hi": float(np.exp(hi)), "ratio_mean": float(np.exp(d.mean())), "ratio_min": float(np.exp(d.min())), "ratio_max": float(np.exp(d.max()))})
    return rows


trows = []
for mat in MATS:
    cm = cond_means(D[(mat, "Alldata", MAIN_DATA)])
    for h in sorted(cm["H"].unique()):
        if np.isclose(h, REF_H):
            continue
        for r in curve_ratio(cm, "H", h, REF_H, "Tset"):
            trows.append({"material": MATS[mat], "kind": "厚み", "compare": f"{h} mm / {REF_H} mm", "H": h, **r})
    for r in curve_ratio(cm, "Tset", 190.0, 230.0, "H"):
        trows.append({"material": MATS[mat], "kind": "温度", "compare": "190 ℃ / 230 ℃", **r})
ratio_df = pd.DataFrame(trows)
ratio_df.to_csv(os.path.join(OUT_DIR, "thickness_ratio.csv"), index=False, encoding="utf-8-sig")
print(ratio_df.round(3).to_string(index=False))
for mat in MATS:
    s = ratio_df[(ratio_df["material"] == MATS[mat])]
    th, tt = s[s["kind"] == "厚み"], s[s["kind"] == "温度"]
    th05 = th[np.isclose(th["H"], 0.5)]["ratio_mean"]
    judge(bool((th["ratio_mean"] - 1).abs().max() > 0.1), f"{MATS[mat]}：同じ γ̇・同じ設定温度で、厚みによる粘度の比が 1 から 10% 以上ずれる（範囲 {th['ratio_mean'].min():.2f}〜{th['ratio_mean'].max():.2f} 倍。0.5 mm / {REF_H} mm は {th05.min():.2f}〜{th05.max():.2f} 倍）→ 見かけの流動曲線は、厚みで重ならない（理想的なスリット流れではない）。温度の比（190/230 ℃）は {tt['ratio_mean'].min():.2f}〜{tt['ratio_mean'].max():.2f} 倍")

In [ ]:
# セル4.5：図1 厚みごとの見かけの流動曲線（条件平均。設定温度ごと）、図2 厚みの比と温度の比
HC = {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"}
fig, axes = plt.subplots(len(MATS), 3, figsize=(15, 4.6 * len(MATS)), squeeze=False)
for i, mat in enumerate(MATS):
    cm = cond_means(D[(mat, "Alldata", MAIN_DATA)])
    for j, t in enumerate(sorted(cm["Tset"].unique())):
        ax = axes[i, j]
        for h, g in cm[cm["Tset"] == t].groupby("H"):
            g = g.sort_values("ln_gd")
            ax.plot(np.exp(g["ln_gd"]), np.exp(g["ln_eta"]), "o-", color=HC[h], ms=5, label=f"{h} mm")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"見かけのせん断速度 $\dot{\gamma}$ (1/s)")
        ax.set_ylabel("見かけの粘度 η (Pa·s)")
        ax.set_title(f"{MATS[mat]}　設定温度 {int(t)} ℃")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(title="厚み")
fig.suptitle("図1　厚みごとの見かけの流動曲線（条件平均）。理想的なスリット流れなら、厚みによらず 1 本に重なる", y=1.01)
fig.tight_layout()
print(save_fig(fig, OUT_DIR, "fig1_flow_curves_by_thickness"))
plt.show()

fig, axes = plt.subplots(1, len(MATS), figsize=(6.5 * len(MATS), 4.6), squeeze=False)
for i, mat in enumerate(MATS):
    ax = axes[0, i]
    s = ratio_df[(ratio_df["material"] == MATS[mat]) & (ratio_df["kind"] == "厚み")]
    for t, g in s.groupby("Tset"):
        ax.errorbar(g["H"] + (t - 210) / 400, g["ratio_mean"], yerr=[g["ratio_mean"] - g["ratio_min"], g["ratio_max"] - g["ratio_mean"]], fmt="o", capsize=3,
                    color={190: "tab:blue", 210: "tab:green", 230: "tab:red"}[int(t)], label=f"厚み：{int(t)} ℃（η(H)/η({REF_H} mm)）")
    tt = ratio_df[(ratio_df["material"] == MATS[mat]) & (ratio_df["kind"] == "温度")]
    ax.axhspan(tt["ratio_mean"].min(), tt["ratio_mean"].max(), color="gray", alpha=0.25, label="温度：η(190 ℃)/η(230 ℃) の範囲")
    ax.axhline(1, color="k", lw=0.8)
    ax.set_xlabel("厚み H (mm)")
    ax.set_ylabel("同じ γ̇ での粘度の比")
    ax.set_title(MATS[mat])
    ax.legend(fontsize=8)
fig.suptitle(f"図2　同じ γ̇・同じ設定温度での、厚みによる粘度の比（点は平均、棒は重なる範囲での最小〜最大）。灰色の帯は、温度の効果", y=1.03)
fig.tight_layout()
print(save_fig(fig, OUT_DIR, "fig2_thickness_ratio"))
plt.show()

## 5. GPR で、H を外す・替えると、どうなるか

モデル（冒頭の表）を、手順3d の 2 つの版で学習・評価する。5data 版：S_V（外挿）、S_T（210 ℃）。Alldata 版：CV（条件単位の 10 分割）。

**見る点**
- **リークの目印**：リークがあると、検証の RMSE が、純粋誤差（約 0.04）より明らかに小さくなる（ショットごとのゆれまで当ててしまう）。H は、同じ条件のショットで同じ値なので、ショットごとのゆれの情報を持たない。
- **H が要るか**：R3 と W0（H を外す）の差。W0 の誤差が、厚みごとに偏るか（厚みごとの残差の平均）。
- **Tff で代わりになるか**：W1（γ̇, Tff）、W1T（γ̇, Tff, Tset）。
- **Fo は H と同じ情報か**：RFo と R3 の差（構造 A・B の入力 ln Fo には、H が入っている）。

In [ ]:
# セル5：学習（並列。15〜25 分）
SPECS = {"R3": ["ln_gd_cm", "ln_H", "Tset"], "RFo": ["ln_gd_cm", "ln_Fo_cm", "Tset"], "W0": ["ln_gd_cm", "Tset"], "W1": ["ln_gd_cm", "ln_Tff_cm"], "W1T": ["ln_gd_cm", "ln_Tff_cm", "Tset"]}
for k in KEYS:
    MODELS[k] = {"kind": "gpr", "cols": SPECS[k], "linear": True}
LABELS = {"R3": r"R3：［$\dot{\gamma}$, $H$, $T_{set}$］", "RFo": r"RFo：［$\dot{\gamma}$, Fo, $T_{set}$］", "W0": r"W0：［$\dot{\gamma}$, $T_{set}$］（Cross-WLF と同じ変数）",
          "W1": r"W1：［$\dot{\gamma}$, $T_{ff}$］（Cross-WLF と同じ変数）", "W1T": r"W1T：［$\dot{\gamma}$, $T_{ff}$, $T_{set}$］"}


def job(mat, ver, name, tr, tests, key, restarts):
    warnings.simplefilter("ignore")
    t0 = time.time()
    fit = fit_model(key, tr, restarts)
    return mat, ver, name, key, fit, {tn: predict_model(key, fit, te)[:3] for tn, te in tests.items()}, time.time() - t0


SETS3E, JOBS = {}, []
for ver in DATA_VERSIONS:
    for mat in MATS:
        vs = version_sets(mat, ver, MAIN_DATA, TEMP_HOLDOUT)
        sp = {k: v for k, v in vs["splits"].items() if (k in ("S_V", "S_T") if ver == "5data" else k.startswith("CV"))}
        SETS3E[(mat, ver)] = {"pool": vs["pool"], "splits": sp}
        for key in KEYS:
            for name, (tr, tests) in sp.items():
                JOBS.append((mat, ver, name, tr, tests, key, N_RESTARTS_CV if name.startswith("CV") else N_RESTARTS))
            if ver == "Alldata":
                JOBS.append((mat, ver, "ALL", vs["pool"], {}, key, N_RESTARTS))      # 最終モデル（高解像度のグラフ用）。5data 版は S_V と同じ学習
JOBS.sort(key=lambda j: -len(j[3]))
t0 = time.time()
RES = Parallel(n_jobs=N_JOBS)(delayed(job)(*j) for j in JOBS)
print(f"{len(JOBS)} 回の学習を並列（{N_JOBS}）で計算：{(time.time() - t0) / 60:.1f} 分")

In [ ]:
# セル6：指標、厚みごとの残差、ハイパーパラメータ
FIT3E, acc, KROWS = {}, {}, []
for (mat, ver, name, key, fit, preds, sec), j in zip(RES, JOBS):
    tr, tests = j[3], j[4]
    FIT3E[(mat, ver, name, key)] = fit
    KROWS.append({"material": MATS[mat], "version": ver, "split": name, "model": key, "n_train": len(tr), **kernel_row(key, fit), "pure_sd_ln_eta": within_stats(tr)["sd_ln_eta"], "fit_sec": sec})
    for tn, te in tests.items():
        mu, sd, sdf = preds[tn]
        split = "CV" if tn.startswith("CV") else tn
        acc.setdefault((mat, ver, split, key), []).append(pd.DataFrame({"y": te["ln_eta"].to_numpy(), "mu": mu, "sd": sd, "H": te["H"].to_numpy(), "Tset": te["Tset"].to_numpy(),
                                                                        "v": te["v"].to_numpy(), "cond": te["cond"].to_numpy(), "group": te["group"].to_numpy()}))
for mat in MATS:
    for key in KEYS:
        if (mat, "5data", "S_V", key) in FIT3E:
            FIT3E[(mat, "5data", "ALL", key)] = FIT3E[(mat, "5data", "S_V", key)]
kern_df = pd.DataFrame(KROWS)
kern_df.to_csv(os.path.join(OUT_DIR, "kernels.csv"), index=False, encoding="utf-8-sig")
PRED3E, mrows, brows = {}, [], []
for (mat, ver, split, key), parts in acc.items():
    d = pd.concat(parts, ignore_index=True)
    if split == "CV":
        d["group"] = np.where(d["v"] <= 5, "低速側", np.where(d["v"] >= 280, "高速側", "内側"))
    PRED3E[(mat, ver, split, key)] = d
    k = kern_df[(kern_df["material"] == MATS[mat]) & (kern_df["version"] == ver) & (kern_df["model"] == key)]
    k = k[k["split"].str.fullmatch(r"CV\d+")] if split == "CV" else k[k["split"] == {"S_T3": "S_T"}.get(split, split)]
    subs = {"全体": np.ones(len(d), bool), **{g: (d["group"] == g).to_numpy() for g in d["group"].unique() if g != "全体"}}
    for sub, m in subs.items():
        if m.sum() < 2:
            continue
        ev = evaluate(d["y"].to_numpy()[m], d["mu"].to_numpy()[m], d["sd"].to_numpy()[m])
        mrows.append({"material": MATS[mat], "version": ver, "split": split, "subset": sub, "model": key, "n_test": int(m.sum()), **ev,
                      "noise_sd": float(k["noise_sd"].mean()), "pure_sd": float(k["pure_sd_ln_eta"].mean())})
    r = d.assign(r=d["y"] - d["mu"])
    for h, g in r.groupby("H"):
        brows.append({"material": MATS[mat], "version": ver, "split": split, "model": key, "H": h, "bias": float(g["r"].mean()), "sd": float(g["r"].std()), "n": len(g)})
met_df = pd.DataFrame(mrows)
bias_df = pd.DataFrame(brows)
met_df.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding="utf-8-sig")
bias_df.to_csv(os.path.join(OUT_DIR, "bias_by_thickness.csv"), index=False, encoding="utf-8-sig")
for mat in MATS:
    print(f"\n【{MATS[mat]}】RMSE（括弧は被覆率）")
    t = met_df[met_df["material"] == MATS[mat]].copy()
    t["v"] = t["RMSE"].map("{:.3f}".format) + "(" + t["coverage95"].map("{:.2f}".format) + ")"
    print(t.pivot_table(index=["version", "split", "subset"], columns="model", values="v", aggfunc="first").reindex(columns=KEYS).to_string())
print("\n【厚みごとの残差の平均（実測 − 予測。ln η）】Alldata 版の CV と、5data 版の S_V")
show = bias_df[((bias_df["version"] == "Alldata") & (bias_df["split"] == "CV")) | ((bias_df["version"] == "5data") & (bias_df["split"] == "S_V"))]
print(show.pivot_table(index=["material", "version", "split", "H"], columns="model", values="bias").reindex(columns=KEYS).round(3).to_string())
print("\n【長さスケール（標準化した入力。小さいほど、その入力で出力が大きく変わる。下限 0.01）】5data 版の S_V")
ls_cols = [c for c in kern_df.columns if c.startswith("ls_")]
print(kern_df[(kern_df["version"] == "5data") & (kern_df["split"] == "S_V")][["material", "model", "noise_sd", "pure_sd_ln_eta"] + ls_cols].round(3).to_string(index=False))

In [ ]:
# セル7：判定
def mv(mat, ver, split, model, sub="全体", col="RMSE"):
    r = met_df[(met_df["material"] == MATS[mat]) & (met_df["version"] == ver) & (met_df["split"] == split) & (met_df["model"] == model) & (met_df["subset"] == sub)]
    return float(r[col].iloc[0]) if len(r) else np.nan


CHECKS = [(v, s) for v, s in [("5data", "S_V"), ("5data", "S_T"), ("Alldata", "CV")] if v in DATA_VERSIONS]
for mat in MATS:
    n = MATS[mat]
    for ver, sp in CHECKS:
        r3, pure = mv(mat, ver, sp, "R3"), mv(mat, ver, sp, "R3", col="pure_sd")
        judge(r3 > 0.8 * pure, f"{n}・{ver} 版 {sp}：H を入れた R3 の RMSE {r3:.3f} が、純粋誤差 {pure:.3f} の 0.8 倍より大きい（ショットごとのゆれまで当ててはいない ＝ H によるリークの目印はない）")
    for ver, sp in CHECKS:
        if "W0" in KEYS and "R3" in KEYS:
            r3, w0 = mv(mat, ver, sp, "R3"), mv(mat, ver, sp, "W0")
            judge(w0 < 1.5 * r3, f"{n}・{ver} 版 {sp}：H を外した W0 の RMSE {w0:.3f} が、R3 {r3:.3f} の 1.5 倍未満（Cross-WLF と同じ変数でも足りる）。超えれば、見かけの粘度の予測に H が要る")
        if "W1" in KEYS and "R3" in KEYS:
            r3, w1 = mv(mat, ver, sp, "R3"), mv(mat, ver, sp, "W1")
            judge(w1 < 1.5 * r3, f"{n}・{ver} 版 {sp}：H の代わりに Tff を使った W1 の RMSE {w1:.3f} が、R3 {r3:.3f} の 1.5 倍未満（流れの中の樹脂温度で、厚みの効果を説明できる）")
    if "RFo" in KEYS and "Alldata" in DATA_VERSIONS:
        r3, rf = mv(mat, "Alldata", "CV", "R3"), mv(mat, "Alldata", "CV", "RFo")
        judge(abs(rf / r3 - 1) < 0.05, f"{n}：Fo を使った RFo の CV の RMSE {rf:.3f} が、R3 {r3:.3f} とほぼ同じ（±5%。Fo は H と同じ情報を持つ ＝ 構造 A・B も、Fo を通して H を使っている）")
    if "W0" in KEYS and "Alldata" in DATA_VERSIONS:
        b = bias_df[(bias_df["material"] == n) & (bias_df["version"] == "Alldata") & (bias_df["split"] == "CV") & (bias_df["model"] == "W0")]
        print(f"   {n}：W0 の厚みごとの残差の平均（Alldata 版の CV）：" + "、".join(f"{h} mm {v:+.3f}" for h, v in zip(b["H"], b["bias"])))

## 6. 図

- 図3：モデルごとの RMSE（分割ごと）。破線は純粋誤差。
- 図4：厚みごとの残差の平均（Alldata 版の CV）。0 に近いほど、厚みによる偏りがない。

In [ ]:
# セル8：図3 RMSE、図4 厚みごとの残差の平均
CATS = [(v, s, sub) for v, s, sub in [("5data", "S_V", "低速側"), ("5data", "S_V", "高速側"), ("5data", "S_T", "全体"), ("Alldata", "CV", "全体"), ("Alldata", "CV", "低速側"), ("Alldata", "CV", "高速側")] if v in DATA_VERSIONS]
MCOL = {"R3": "k", "RFo": "tab:gray", "W0": "tab:red", "W1": "tab:orange", "W1T": "tab:purple"}
fig, axes = plt.subplots(1, len(MATS), figsize=(7.5 * len(MATS), 4.8), squeeze=False)
for i, mat in enumerate(MATS):
    ax = axes[0, i]
    w = 0.8 / len(KEYS)
    for k, key in enumerate(KEYS):
        ax.bar(np.arange(len(CATS)) + (k - (len(KEYS) - 1) / 2) * w, [mv(mat, v, s, key, sub) for v, s, sub in CATS], w, color=MCOL[key], label=LABELS[key])
    pure = np.nanmean([mv(mat, v, s, "R3", sub, "pure_sd") for v, s, sub in CATS])
    ax.axhline(pure, color="k", ls="--", lw=0.8, label=f"純粋誤差（約 {pure:.3f}）")
    ax.set_xticks(np.arange(len(CATS)))
    ax.set_xticklabels([f"{v} 版\n{s} {sub}" for v, s, sub in CATS], fontsize=8)
    ax.set_ylabel("RMSE（ln η）")
    ax.set_title(MATS[mat])
    ax.legend(fontsize=7)
fig.suptitle("図3　モデルごとの RMSE。H を外す（W0・W1）と、誤差が大きくなる", y=1.02)
fig.tight_layout()
print(save_fig(fig, OUT_DIR, "fig3_rmse_by_model"))
plt.show()

if "Alldata" in DATA_VERSIONS:
    fig, axes = plt.subplots(1, len(MATS), figsize=(6.5 * len(MATS), 4.6), squeeze=False)
    for i, mat in enumerate(MATS):
        ax = axes[0, i]
        b = bias_df[(bias_df["material"] == MATS[mat]) & (bias_df["version"] == "Alldata") & (bias_df["split"] == "CV")]
        for k, key in enumerate(KEYS):
            g = b[b["model"] == key].sort_values("H")
            ax.errorbar(g["H"] + (k - 2) * 0.03, g["bias"], yerr=g["sd"] / np.sqrt(g["n"]), fmt="o-", color=MCOL[key], capsize=3, label=LABELS[key])
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xlabel("厚み H (mm)")
        ax.set_ylabel("残差の平均（実測 − 予測、ln η）")
        ax.set_title(f"{MATS[mat]}（Alldata 版の CV）")
        ax.legend(fontsize=7)
    fig.suptitle("図4　厚みごとの残差の平均。H を入れたモデル（R3・RFo）は 0 の上に並び、H を外したモデルは厚みで偏る", y=1.03)
    fig.tight_layout()
    print(save_fig(fig, OUT_DIR, "fig4_bias_by_thickness"))
    plt.show()

## 7. 高解像度のグラフ（報告書・発表資料用）

せん断速度 ― 粘度の学習結果グラフを、GPR6月ノートのセル9-2 と同じ見せ方で、**600 dpi（15 × 10 インチ）**で 1 枚ずつ保存する。線は予測の平均、帯は 95% 区間、塗りつぶしの点は学習に使った条件、白抜きの点は学習に使わなかった条件（検証）、× は問題データ一覧の「感度確認」の条件。

- **レイアウト**：`"thickness"`＝厚みごとに 1 枚（3 つの設定温度の曲線）、`"temperature"`＝設定温度ごとに 1 枚（4 つの厚みの曲線。6月のノートと同じ）。
- **変えられるもの**（セルの先頭の設定エリア）：描くモデル・材料・学習の種類、帯（予測区間／信頼区間）、点（条件平均／ショットごと）、軸の範囲、予測線の延長、図の大きさ・解像度・文字・マーカーの大きさ、タイトルと軸名の表示、保存形式（png／pdf／svg）。
- 全体を一度実行して、学習が終わった後なら、このセルだけを何度でも実行し直せる。

H を外したモデル（W0・W1）は、厚みが違っても同じ曲線になる（入力に H がないため）。設定温度ごとのレイアウトで、4 つの厚みの曲線が重なり、点が曲線から離れる様子が見える。

In [ ]:
# セル9：高解像度のグラフ（せん断速度 ― 粘度。GPR6月ノートのセル9-2 にならう）
# ==========================================
# ▼▼ 設定エリア（グラフの見た目・文字サイズ。このセルの設定だけを直して、このセルだけを実行し直せる） ▼▼
# ==========================================
HR_MODELS = ["R3", "W0", "W1"]     # 描くモデル（KEYS のどれでも）
HR_VERSIONS = ["Alldata"]          # 描く版（"5data"：5data で学習〔3data は白抜き〕／"Alldata"：全条件で学習）
HR_MATERIALS = ["PP", "MIX"]       # 描く材料（PP：PP、MIX：粉砕材混合 PP）
HR_LAYOUTS = ["thickness", "temperature"]   # "thickness"：厚みごとに 1 枚（3 つの設定温度）／"temperature"：設定温度ごとに 1 枚（4 つの厚み。6月のノートと同じ）
HR_BAND = "predict"                # 帯："predict"＝95% 予測区間（ショット間のばらつき込み）／"latent"＝95% 信頼区間（平均の不確かさだけ）／None＝帯なし
HR_POINTS = "cond"                 # 点："cond"＝条件平均／"shot"＝ショットごと
HR_SHOW_TEST = True                # 学習に使わなかった条件（検証）を、白抜きで重ねるか
HR_MARK_SENS = True                # 問題データ一覧の「感度確認」の条件を、× で重ねるか
HR_X_RANGE = None                  # 予測線の範囲 [1/s]。None なら、各線のデータの範囲を HR_EXTEND 倍に延長。(30, 50000) のように指定すると、全部の線をその範囲に延長（外挿の確認用）
HR_EXTEND = 1.3
HR_XLIM = {"thickness": None, "temperature": (30, 50000)}   # 横軸（せん断速度 [1/s]）の表示範囲。(下限, 上限)＝固定（10 の累乗の目盛り。6月と同じ）／None＝その図のデータに合わせる（1・2・5 の目盛り）。
HR_YLIM = {"thickness": None, "temperature": (0.5, 5000)}   # 縦軸（粘度 [Pa·s]）の表示範囲。同じ書き方。レイアウトごとに変えられる（全部を同じ範囲にするなら、(30, 50000) のように書く）
HR_FIGSIZE = (15, 10)              # 図の大きさ（幅, 高さ）[インチ]
HR_DPI = 600                       # 解像度
HR_FONT = {"axis": 24, "legend": 24, "title": 24, "tick": 28}   # 軸名・凡例・タイトル・目盛り数字の文字サイズ
HR_MARKER = 200                    # 点の大きさ（条件平均のとき。面積）
HR_MARKER_SHOT = 70                # 点の大きさ（ショットごとのとき）
HR_LINE = 3.0                      # 予測線の太さ
HR_SHOW_TITLE = True               # タイトルを付けるか
HR_SHOW_XLABEL = False             # 軸名を付けるか（6月のノートと同じく、既定は付けない。資料に貼るとき、別に付ける場合のため）
HR_SHOW_YLABEL = False
HR_FORMATS = ["png"]               # 保存形式。"png"・"pdf"・"svg" から選ぶ（pdf・svg は、拡大しても粗くならない）
HR_SHOW = False                    # True なら、画面にも表示する（重い）
# ==========================================
HR_KW = dict(layouts=HR_LAYOUTS, band=HR_BAND, points=HR_POINTS, show_test=HR_SHOW_TEST, mark_sens=HR_MARK_SENS, x_range=HR_X_RANGE, extend=HR_EXTEND,
             xlim=HR_XLIM, ylim=HR_YLIM, figsize=HR_FIGSIZE, dpi=HR_DPI, font=HR_FONT, marker=HR_MARKER, marker_shot=HR_MARKER_SHOT, line=HR_LINE,
             show_title=HR_SHOW_TITLE, show_xlabel=HR_SHOW_XLABEL, show_ylabel=HR_SHOW_YLABEL, formats=HR_FORMATS, show=HR_SHOW)
HR_DIR = os.path.join(OUT_DIR, "highres")

hr_saved = []
for mat in HR_MATERIALS:
    for ver in HR_VERSIONS:
        for key in HR_MODELS:
            if (mat, ver, "ALL", key) not in FIT3E:
                print(f"{MATS[mat]}・{ver} 版・{key} は、学習していないので描かない（DATA_VERSIONS・KEYS に入れて、上から実行し直す）")
                continue
            t0 = time.time()
            fit = FIT3E[(mat, ver, "ALL", key)]
            pool = SETS3E[(mat, ver)]["pool"] if ver == "Alldata" else pool_of(mat, MAIN_DATA)
            hr_saved += hr_flow(lambda Xg, f=fit, k=key: predict_model(k, f, Xg.assign(ln_Fo_cm=-(Xg["ln_gd_cm"] + 3 * Xg["ln_H"])))[:3], mat, MAIN_DATA,
                                set(SETS3E[(mat, ver)]["pool"]["cond"]), f"{LABELS[key]}（{ver} 版）", f"hr_{mat}_{key}_{ver}", HR_DIR, pool=pool, **HR_KW)
            print(f"{MATS[mat]}・{ver} 版・{key}：保存したファイル 累計 {len(hr_saved)} 個（{time.time() - t0:.0f} 秒）")
judge(len(hr_saved) > 0, f"高解像度のグラフを {len(hr_saved)} 個保存した（{os.path.normpath(HR_DIR)}）。形式 {HR_FORMATS}、{HR_DPI} dpi")

In [ ]:
# セル10：保存したファイル
print("保存したファイル:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", os.path.normpath(os.path.join(OUT_DIR, f)))